In [5]:
import os
from openai import OpenAI




openai_api_key = "sk-proj"
client = OpenAI(
    # This is the default and can be omitted
    api_key=openai_api_key
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "system", "content": "Hello! How can I assist you today?",
            "role": "user", "content": "",
        }
    ],
    model="gpt-4o-mini",
    max_tokens=10
)

print(chat_completion.choices[0].message)

ChatCompletionMessage(content='Hello! How can I assist you today?', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
path = "/content/drive/MyDrive/Clean Transcripts"

In [ ]:
import chardet

folder_path = path

transcripts = {}

count = 0
print(f'There are {len(sorted(os.listdir(folder_path)))} files for our total transcript')
# Iterate through each file in the specified folder
for filename in sorted(os.listdir(folder_path)):
    # Check if the file is a .txt file
    if not filename.endswith('.txt'):
        print(f'{filename} is not a text file')
        #raise ValueError(f'Non txt file: {filename} in directory!')
    else:
        # Construct the full file path
        file_path = os.path.join(folder_path, filename)
        # Open the file and read its contents into a string

        with open(file_path, 'rb') as f:
            result = chardet.detect(f.read())
            encoding = result['encoding']
        # print(f'{filename}, {encoding}')

        with open(file_path, 'r', encoding = encoding) as file:
            transcript = file.read()

        transcripts[filename.split('.')[0]] = transcript

        count += 1


print(count)

There are 272 files for our total transcript
272


In [ ]:
def summarize_transcipt(transcript:str) -> str:
    chat_completion = client.chat.completions.create(
        messages=[
            {"role": "system", "content": '''Read the following call transcript between a provider and a patient, generate a detailed summary in 4 following sections:

            Patient Information: Key details about the patient discussed in the call.

            Symptoms Discussed: A list of symptoms the patient reported.

            Diagnosis and Assessment: The provider’s assessment and diagnosis of the patient’s condition.

            Treatment Plan: Details of the treatment plan or medications prescribed.
            '''},
            {"role": "user", "content": transcript}
        ],
        model="gpt-4o-mini",
        max_tokens=1000
    )

    return chat_completion.choices[0].message.content

In [ ]:
summaries = {}

for key in transcripts:
    # Call the pre-defined function and store the result
    summary = summarize_transcipt(transcripts[key])

    print(key)
    print(summary)
    print(len(summary.split()))
    print()
        # Store the summary in the dictionary using the file name as the key
    filename=key.split('.')[0]
    summaries[filename] = summary

Streaming output truncated to the last 5000 lines.
### Treatment Plan:
- A comprehensive physical examination is to be conducted during an in-person appointment.
- Laboratory work will be ordered, including:
  - Complete Blood Count (CBC)
  - C-reactive protein (CRP) and Erythrocyte Sedimentation Rate (ESR) for inflammatory markers.
  - Rheumatoid factor and antinuclear antibodies testing for rheumatological assessment.
- Imaging will include X-rays of the spine, sacroiliac joints, hands, and knees to evaluate for arthritis or other potential structural issues.
- The provider will follow up with the patient shortly to schedule the necessary examinations and tests.
338

MSK0004
### Patient Information:
The patient is a male software engineer who lives alone in an apartment. He is preparing for an upcoming race and has been training regularly. He is currently not taking any prescription medications, other than Vitamin D and Vitamin A supplements, and has no known allergies or significant

In [ ]:
import pandas as pd
df=pd.DataFrame(summaries.items(), columns=['File Name', 'Summary'])
df.to_csv('new_summaries.csv', index='File Name')
df

,File Name,Summary
0,CAR0001,### Patient Information\nThe patient is a 39-y...
1,CAR0002,### Patient Information:\nThe patient is a mal...
2,CAR0003,**Patient Information:**\nThe patient is a mid...
3,CAR0004,### Patient Information\n- **Age**: 25 years o...
4,CAR0005,**Patient Information:** \nThe patient is an ...
...,...,...
267,RES0213,**Patient Information:** \nThe patient is a 2...
268,RES0214,### Patient Information\n- **Age**: Approximat...
269,RES0215,### Patient Information:\nThe patient is an 8-...
270,RES0216,### Patient Information:\n- **Patient Name**: ...


In [ ]:
df=pd.read_csv('/content/new_summaries.csv')
df.head()
df.loc[df['File Name']=='RES0051']['Summary'].values[0]

'**Patient Information:**  \nThe patient is a 12-year-old (grade 6) individual living with their parents and two younger siblings. They do not have any significant past medical history or ongoing medical conditions. The patient has no known allergies except possibly penicillin, which caused a rash in the past. They do not take any medications and have not had surgeries or hospitalizations.\n\n**Symptoms Discussed:**  \nThe patient reported the following symptoms:  \n- Persistent cough for the past five days, worsening over time  \n- Sore throat, which started around the same time as the cough  \n- Altered taste sensation for the last two days  \n- Minimal fatigue (feeling a little tired)  \n- Possible fever (feeling hot at night but no measured temperature)  \n- No phlegm, blood, or nasal discharge  \n- No difficulties in breathing, chest pain, or gastrointestinal issues  \n- No urinary problems, muscle aches, rashes, or other significant systemic symptoms  \n\n**Diagnosis and Assessme

In [ ]:
df.head()

,Unnamed: 0,File Name,Summary
0,0,CAR0001,### Patient Information\nThe patient is a 39-y...
1,1,CAR0002,### Patient Information:\nThe patient is a mal...
2,2,CAR0003,**Patient Information:**\nThe patient is a mid...
3,3,CAR0004,### Patient Information\n- **Age**: 25 years o...
4,4,CAR0005,**Patient Information:** \nThe patient is an ...
